# Pulsar Search Steps

Pulsars are periodic, dispersed radio signals.  The largest problem in finding them is that they are rather faint compared to human-made radio interference (which is prevalent, even in 'radio-quiet' zones)

A pulsar search roughly follows these steps:

- Finding and cleaning RFI 
- De-dispersing 
- Fourier transforming in time 
- Remove more RFI: periodic peaks at DM=0 (Earth-sources are not typically dispersed)
- Searching for Peaks in  spin frequency, DM
- 'Folding' at possible spin peaks
- Inspecting, does this look like a pulsar?

The goal of this tutorial is to find the pulsar, and determine its spin period and DM.  

## Import the necessary modules:
- numpy for management of 2D data arrays
- matplotlib for plotting
- astropy for units and unit conversion

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

## Load the data
The cell below has the basic commands to load the data, the time sampling and observing frequencies of each channel in the data. 

In [ ]:
filename = '../data/335.13_48.05_60195.29196.npy'
bad_channels = np.load('../data/badchannellist.npy')
data = np.load(filename)
data_clean = np.copy(data)
data_clean[:, bad_channels] = 0

# if you want to make your life easy to start, just use a small snippet of data.  e.g. the below is ~8seconds
# data = data[:4096]

tsamp = 2*0.98304*u.ms
frequencies = np.linspace(800,400,512, endpoint=False)*u.MHz
times = np.arange(data.shape[0]) * tsamp


# Visualize the data
For a bright pulsar, you may be abe to see every individual dispersed pulse

A good place to start is to visualize a small subset of the data, to get a sense of how it looks, and the Radio Frequency Interference (RFI) in your dataset

In [ ]:
# You can subtract or divide the time-averaged background to visualize
data_bgsub = data / np.nanmean(data, axis=0) - 1 #...
data_bgsub[np.isnan(data_bgsub)] = 0

plt.figure(figsize=(12, 6))
plt.title('Raw data')
plt.imshow(data[:1000].T, aspect='auto', interpolation='nearest')
plt.colorbar()
plt.xlabel('time (samples)')
plt.ylabel('frequency (channels)')


plt.figure(figsize=(12,6))
plt.title('BG subtracted')

plt.imshow(data_bgsub[:1000].T, aspect='auto', interpolation='nearest', vmin=-0.2, vmax=0.2)
plt.colorbar()
plt.xlabel('time (samples)')
plt.ylabel('frequency (channels)')


data_bgsub = data_clean / np.nanmean(data_clean, axis=0) - 1 #...
data_bgsub[np.isnan(data_bgsub)] = 0

plt.figure(figsize=(12,6))
plt.title('Cleaned, BG subtracted')

plt.imshow(data_bgsub[:1000].T, aspect='auto', interpolation='nearest', vmin=-0.2, vmax=0.2)
plt.colorbar()
plt.xlabel('time (samples)')
plt.ylabel('frequency (channels)')

# Search across dispersion measure

To find a pulsar at an unknown DM, you must dedisperse at many trial DMs

A simple algorithm is as follows:

- Choose a search grid of DMs, from 0 (terrestrial), to the max DM you expect
- Create a fine enough grid so that pulses are guaranteed to align (often dDM = 1 time sample)
- For each DM trial:
    - For each frequency:
        - compute the time shift, in units of samples
        - shift / roll your data
    - Sum the de-dispersed data across frequency
    - I(DM_trial, t)
        
The final array is the frequency summed power I(t), at each DM_trial

In [ ]:
# Here I search out to DM=100pc/cm^3 - well above the maximum in the MW at high Galactic latitude
DM_trials = np.linspace(0, 100, 200)*u.pc/u.cm**3

# dispersion delay constant
k_DM = (1/2.41e-4)  *u.cm**3 / u.pc * u.MHz**2 *u.s

# reference to top of band
f_ref = 800*u.MHz 

# I(time, DM)
IDMt = np.zeros((data_clean.shape[0], len(DM_trials)))

for i, DMi in enumerate(DM_trials):
    t_delays = k_DM * DMi * (1./f_ref**2 - 1./frequencies**2)
    t_delays = t_delays.to(u.s)
    bin_shifts = (t_delays / tsamp).decompose().astype('int')
    data_DD = np.copy(data_clean)
    
    for ifreq in range(len(frequencies)):
        data_DD[:, ifreq] = np.roll(data_clean[:, ifreq], bin_shifts[ifreq])
    IDMt[:, i] = data_DD.sum(-1)
    
    
# visualize
plt.figure(figsize=(12,4))
plt.pcolormesh(times.value/1000., DM_trials.value, IDMt.T)
plt.colorbar()

plt.xlabel('time (s)')
plt.ylabel('DM (pc / cm^3)')

# Blindly finding pulsars

After de-dispersing, you have the power as I(time, DM)

If the pulsar is bright enough to see in single pulses, you will see a peak at the time of the pulse, and at its DM - a pulsar discovery!  (this is how FRBs are found, and Rotating Radio Transients, "RRATs".

However, often a pulsar is too faint to see in single pulses.  Then it can be found through a periodicity search, either:
- FFT along the time axis, or..
- Folding at many trial periods.

If you take an FFT, you will be sensitive to frequencies up to 1/ (2tsamp).  For ~2ms time samples, this is a maximum of 250Hz.  For a regular ~1Hz pulsar, you would expect to see it and a range of harmonics.  To increase S/N, and precision on your measurement of P_spin, you can sum harmonics.  For now, try to find the peak in DM, F_spin

Reminder: noise, and terrestrial signals are expected at DM=0
Periodic signals at DM=0 are often called 'birdies', while the increase of power at low freuqencies is 'red noise'

In [ ]:
PS = np.abs(np.fft.rfft(IDMt, axis=0))**2
PS = PS / np.median(PS, axis=0, keepdims=True)
spin_freq = np.fft.rfftfreq(IDMt.shape[0], d=tsamp).to(u.Hz)

plt.figure(figsize=(12, 6))
plt.pcolormesh(spin_freq.value, DM_trials.value, np.log10(PS).T, vmin=0, vmax=2)

plt.xlim(0, 64)
plt.ylabel('DM (pc / cm^3)')
plt.xlabel('Spin Frequency (Hz)')

# 'Folding'

To fold a pulsar, you take a spin period, and map the time axis to spin phase as

phase = (t%P_spin) // P_spin

If you fold with the correct period, it will line up in phase.
If you are completely wrong, the pulsar will wrap in phase over time, and average out
If you are slightly off, the pulse will slowly drift in phase over time

Try!  You can use the DM, P_spin determined from the FFT search, or that you determined visually

In [ ]:
def fold_pulsar(timeseries, tsamp, P_spin, nbins=128):
    """
    Fold a 1D pulsar time series into I(pulse_number, phase).

    This version performs true phase binning:
    each sample is assigned to a pulse number and phase bin,
    then adjacent samples are summed/averaged together.

    Partial first/last pulses are retained.

    Parameters
    ----------
    timeseries : array_like
        Input intensity timestream.

    tsamp : astropy Quantity
        Sampling time.

    P_spin : astropy Quantity
        Pulsar spin period.

    nbins : int
        Number of phase bins.

    Returns
    -------
    fold : ndarray
        Folded array with shape (n_pulses, nbins)

    phase : ndarray
        Phase axis from 0 to 1.

    pulse_times : Quantity
        Pulse start times.
    """

    tsamp = tsamp.to(u.s)
    P_spin = P_spin.to(u.s)

    data = np.asarray(timeseries)

    # sample times
    t = np.arange(len(data)) * tsamp

    # pulse number for each sample
    pulse_number = np.floor((t / P_spin).decompose().value).astype(int)

    # phase within pulse [0,1)
    phase_frac = ((t / P_spin).decompose().value) % 1.0

    # phase bin index
    phase_bin = np.floor(phase_frac * nbins).astype(int)
    phase_bin = np.clip(phase_bin, 0, nbins - 1)

    n_pulses = pulse_number.max() + 1

    # accumulate intensity
    fold = np.zeros((n_pulses, nbins))
    counts = np.zeros((n_pulses, nbins))

    for i in range(len(data)):
        p = pulse_number[i]
        b = phase_bin[i]

        fold[p, b] += data[i]
        counts[p, b] += 1

    # average occupied bins
    mask = counts > 0
    fold[mask] /= counts[mask]

    # optional: mark empty bins as NaN
    fold[~mask] = np.nan

    phase = np.linspace(0, 1, nbins, endpoint=False)

    pulse_times = np.arange(n_pulses) * P_spin

    return fold, phase, pulse_times

In [ ]:
##### Fold I(t, DM_best) - based off of the best DM, and spin period from your search
P = XYZ*u.ms # best P, astropy unit
DM_best = ABC*u.pc / u.cm**3 # best DM

i_DM = np.argmin( abs(DM_trials-DM_best) )
I_t = IDMt[:, i_DM]

fold, phase, pulse_times = fold_pulsar(
    I_t,
    tsamp=tsamp,
    P_spin=P,
    nbins=128
)
fold_bg = fold - np.median(fold, axis=-1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(6,9),
    sharex=True,
    gridspec_kw={'height_ratios': [1, 2]}
)

# top profile panel
ax1.plot(phase, np.nanmean(fold_bg, axis=0))
ax1.set_xlim(0, 1)

# remove redundant x ticks/labels
ax1.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

# bottom dynamic spectrum panel
pcm = ax2.pcolormesh(
    phase,
    pulse_times,
    fold_bg,
    shading='auto'
)

ax2.set_xlabel("Pulse phase")
ax2.set_ylabel("Time (s)")

# remove vertical spacing
fig.subplots_adjust(hspace=0)